In [1]:
import pandas as pd
import ast
import os

def extract_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
genus_name = 'Escherichia'

acc_bio = pd.read_csv('/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/biosample_info.tsv', sep='\t')
acc_time = pd.read_csv('/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/assembly_submission_info.tsv', sep='\t')
acc_time["submissionDate"] = pd.to_datetime(acc_time["submissionDate"])

target_dir = f'/active-data/analysis_results/chr_pla/genus/statistics_records/{genus_name}'
replicon_data = pd.read_csv(f'{target_dir}/replicon-plasmid_fraction-self_bitscore_statistics.csv')
all_trans = replicon_data[replicon_data['category-pident_90']=='intermediate replicon'].copy()
all_trans['acc_n'] = all_trans['accession'].str.split('-').str[0]
all_acc = set(all_trans['acc_n'])

In [3]:
filted_bio = acc_bio[(acc_bio['accession'].isin(all_acc)) & (acc_bio['srr_list'] != '[]')]
filted_bio = pd.merge(filted_bio, acc_time, how='left', on='accession')
filted_bio = filted_bio.sort_values(by="submissionDate", ascending=False, ignore_index=True)

In [4]:
import os
import shutil
import subprocess
from pathlib import Path
from typing import List, Dict, Tuple

BASE_WORK_ROOT = Path("/active-data/genome_re-assemble")

def make_dirs(dir_list: List[Path]):
    for d in dir_list:
        d.mkdir(parents=True, exist_ok=True)

def run_cmd(cmd: list):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    stdout, stderr = proc.communicate()
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd, output=stdout, stderr=stderr)

def validate_sra(sra_path: Path) -> bool:
    if not sra_path.exists():
        return False
    proc = subprocess.Popen(["vdb-validate", str(sra_path)], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    proc.communicate()
    return proc.returncode == 0

def download_single_srr(srr_id: str, raw_sra_dir: Path):
    sra_sub = raw_sra_dir / srr_id
    sra_file = sra_sub / f"{srr_id}.sra"
    if sra_file.exists():
        if validate_sra(sra_file):
            return
        shutil.rmtree(sra_sub)
    run_cmd(["prefetch", "--max-size", "200G", "-O", str(raw_sra_dir), srr_id])
    if not validate_sra(sra_file):
        raise RuntimeError(f"Failed to verify the download of {srr_id}")

def download_gcf_sra(accession: str, run_info_list: List[Dict]) -> Tuple[Path, Path]:
    work_dir = BASE_WORK_ROOT / accession
    dir_sra = work_dir / "raw_sra"
    dir_fq = work_dir / "raw_fastq"
    make_dirs([work_dir, dir_sra, dir_fq])

    all_srr = [r["run_id"] for r in run_info_list]
    for srr in all_srr:
        download_single_srr(srr, dir_sra)

In [8]:
count = 0

for idx in filted_bio.index:
    short_read, long_read = False, False
    srr_list = ast.literal_eval(filted_bio.loc[idx, 'srr_list'])
    for run in srr_list:
        if run['platform'] == 'ILLUMINA':
            short_read = True
        if run['platform'] == 'OXFORD_NANOPORE' or run['platform'] == 'PACBIO_SMRT':
            long_read = True
    if short_read and long_read:
        print(filted_bio.loc[idx, 'organismName'], filted_bio.loc[idx, 'accession'], filted_bio.loc[idx, 'srr_list'])
        download_gcf_sra(filted_bio.loc[idx, 'accession'], srr_list)
        count += 1

Escherichia coli GCF_049949635.1 [{'run_id': 'SRR32729544', 'platform': 'ILLUMINA', 'layout': 'PAIRED'}, {'run_id': 'SRR32729613', 'platform': 'OXFORD_NANOPORE', 'layout': 'SINGLE'}, {'run_id': 'SRR36889076', 'platform': 'OXFORD_NANOPORE', 'layout': 'SINGLE'}]
Escherichia coli GCF_964200005.1 [{'run_id': 'ERR13363630', 'platform': 'OXFORD_NANOPORE', 'layout': 'SINGLE'}, {'run_id': 'ERR13363302', 'platform': 'ILLUMINA', 'layout': 'PAIRED'}]
Escherichia coli GCF_039604535.2 [{'run_id': 'SRR28994696', 'platform': 'ILLUMINA', 'layout': 'PAIRED'}, {'run_id': 'SRR29882843', 'platform': 'OXFORD_NANOPORE', 'layout': 'SINGLE'}]
Escherichia coli GCF_034643255.1 [{'run_id': 'SRR27089203', 'platform': 'ILLUMINA', 'layout': 'PAIRED'}, {'run_id': 'SRR27089213', 'platform': 'OXFORD_NANOPORE', 'layout': 'SINGLE'}]
Escherichia coli GCF_034120885.1 [{'run_id': 'SRR27050476', 'platform': 'ILLUMINA', 'layout': 'PAIRED'}, {'run_id': 'SRR27285304', 'platform': 'OXFORD_NANOPORE', 'layout': 'SINGLE'}]
Escheri

In [7]:
filted_bio

,accession,organismName,taxId,biosample,srr_list,submissionDate,submitter
0,GCF_051136185.1,Escherichia coli,562,SAMN48718394,"[{'run_id': 'SRR33690897', 'platform': 'ILLUMI...",2025-05-24 06:27:05.403,Hadeer Aly Ibrahim Aly Morgaan
1,GCF_049949635.1,Escherichia coli,562,SAMN47407288,"[{'run_id': 'SRR32729544', 'platform': 'ILLUMI...",2025-03-16 20:27:04.643,Toho University
2,GCF_964200005.1,Escherichia coli,562,SAMEA115834832,"[{'run_id': 'ERR13363630', 'platform': 'OXFORD...",2024-11-02 10:01:11.655,Institut de Recherche pour le Developpement
3,GCF_039604535.2,Escherichia coli,562,SAMN41327117,"[{'run_id': 'SRR28994696', 'platform': 'ILLUMI...",2024-05-10 14:56:04.830,Utah Public Health Laboratory Infectious Disea...
4,GCF_034643255.1,Escherichia coli,562,SAMN38673177,"[{'run_id': 'SRR27089203', 'platform': 'ILLUMI...",2023-12-05 16:47:09.277,The University of Texas at Dallas
5,GCF_034120885.1,Escherichia coli,562,SAMN38646034,"[{'run_id': 'SRR27050476', 'platform': 'ILLUMI...",2023-12-04 18:54:05.026,Utah Public Health Laboratory Infectious Disea...
6,GCF_033395675.1,Escherichia coli,562,SAMN37709341,"[{'run_id': 'SRR26305304', 'platform': 'ILLUMI...",2023-10-06 16:36:05.353,Lawrence Berkeley National Laboratory
7,GCF_032300135.1,Escherichia coli,562,SAMN37300944,"[{'run_id': 'SRR26256955', 'platform': 'PACBIO...",2023-09-06 14:50:04.980,FDA-CVM
8,GCF_030908705.1,Escherichia coli O157:H7,83334,SAMN36938185,"[{'run_id': 'SRR25603930', 'platform': 'ILLUMI...",2023-08-10 17:56:04.623,Utah Public Health Laboratory Infectious Disea...
9,GCF_030908665.1,Escherichia coli O157:H7,83334,SAMN36828866,"[{'run_id': 'SRR25510170', 'platform': 'ILLUMI...",2023-08-03 19:31:05.833,Utah Public Health Laboratory Infectious Disea...
